# Pre-course refresher: R, probability and simulation

**Bayesian statistics for PhD researchers · Aalborg University · 9–17 November 2026**

This refresher gets everyone to the same starting point before Day 1. It is **self-paced** and **not graded** or submitted. It has seven parts:

| Part | Topic |
|---|---|
| 0 | Install the software and check that Stan works |
| 1 | R essentials with a real dataset |
| 2 | Probability: distributions, expectation, conditioning, Bayes' rule |
| 3 | A little calculus: normalising a density, logs, maximising |
| 4 | Simulation and a first look at a likelihood |
| 5 | Your own research question |
| 6 | Self-check |

**This one notebook is all you need.** Everything (explanations, data, exercises, hints) is here; there are no separate scripts or handouts. Open it in JupyterLab with the **R (bayes)** kernel and run the cells in order. Each task has a code cell to complete and an answer cell for your explanation. Hints are under **Hint**. A worked solution notebook (`pre-course-refresher-solutions.ipynb`) is in the same folder: **try first, then check**.

If a part is easy for you, skim it. If one is hard, note it in the self-check (Part 6) and bring the questions to Day 1. Nothing here is a prerequisite you must master; the course builds everything again from the start.

The data are R's built-in `airquality` dataset: daily air-quality measurements in New York, May–September 1973. They are real data, used here only for practice.

In [ ]:
library(tidyverse)   # dplyr for data, ggplot2 for figures
set.seed(2026)
options(width = 100, repr.plot.width = 8, repr.plot.height = 5)
q90 <- function(x) quantile(x, c(.05, .5, .95))

## Part 0 — Software

Follow the course installation guide, `installations.md` (sections 1–3 and 7), or the logistics slides:

1. Install Miniconda.
2. Install the C++ tools: macOS and Linux only.
3. Create the `bayes` environment with R and CmdStan.
4. Add JupyterLab and the **R (bayes)** kernel.

Then run the cell below. It compiles and runs a small Stan model. The first compile can take a minute or two. If it fails, keep the first error message and bring it to Day 1; there is setup help in the first exercise session.

In [3]:
R.version.string
library(cmdstanr)
cmdstan_version()
file <- file.path(tempdir(), "bernoulli.stan")   # copy the example to a folder we can write to
file.copy(file.path(cmdstan_path(), "examples", "bernoulli", "bernoulli.stan"), file, overwrite = TRUE)
mod <- cmdstan_model(file, quiet = TRUE)
fit <- mod$sample(data = list(N = 10, y = c(0, 1, 0, 0, 0, 0, 0, 0, 0, 1)), seed = 123, refresh = 0, show_messages = FALSE)
fit$summary("theta")

[1] "R version 4.5.3 (2026-03-11)"

[1] "2.40.0"

variable,mean,median,sd,mad,q5,q95,rhat,ess_bulk,ess_tail
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
theta,0.2568822,0.2410806,0.1242848,0.1271501,0.08003641,0.4852967,1.000955,1231.553,1476.736


**Your answer:** *(write here)*

## Part 1 — R essentials

### R1. Vectors and missing values (20 min)

(a) Create `x <- c(4, 8, NA, 15, 16, 23, 42)`. Compute its length, the number of observed values, the mean ignoring the missing value, and the mean of the values above 10. (b) Explain in one sentence why `mean(x)` returns `NA`, and why removing missing values is a decision, not a technicality.

In [4]:
x <- c(4, 8, NA, 15, 16, 23, 42)
# length(), sum(!is.na(x)), mean(..., na.rm = TRUE), x[x > 10 & !is.na(x)]

<details><summary><b>Hint</b> (click to open)</summary>

`NA > 10` is `NA`, so add `!is.na(x)` to the condition, or use `which(x > 10)`.

</details>

**Your answer:** *(write here)*

### R2. A data frame (40 min)

`airquality` has columns `Ozone` (ppb), `Solar.R` (langley), `Wind` (mph), `Temp` (°F), `Month` (5–9) and `Day`.

(a) Report the number of rows, the number of missing values in each column, and what one row represents. (b) Create `TempC`, the temperature in °C. (c) Compute the mean, median and SD of `Ozone` for each month, together with the number of observed values. (d) Which month has the highest median ozone, and how many observations is that based on?

In [ ]:
aq <- airquality
str(aq)
# (a) colSums(is.na(aq))
# (b) aq <- aq |> mutate(TempC = ...)
# (c) aq |>
#       filter(!is.na(Ozone)) |>
#       group_by(Month) |>
#       summarise(n = n(), mean = ..., median = ..., sd = ...)

<details><summary><b>Hint</b> (click to open)</summary>

°C = (°F − 32) × 5/9. Read the pipe `|>` as “and then”: take `aq`, and then drop the days without an ozone value (`filter()`), and then split the rows by month (`group_by()`), and then compute one summary row per month (`summarise()`). `n()` counts the rows in each group. For (d), add `arrange(desc(median))` to sort the months.

</details>

**Your answer:** *(write here)*

### R3. Write a function (30 min)

Write a function `describe(v)` that returns the number of observed values, the mean, the SD and the **standard error** of the mean (SD/√n) of a numeric vector, ignoring missing values. Apply it to `Ozone`, `Wind` and `TempC`. Why does the standard error get smaller with more data while the SD does not?

In [ ]:
describe <- function(v) {
  # your code
}

<details><summary><b>Hint</b> (click to open)</summary>

Inside the function, first remove missing values, then compute `n <- length(v)`.

</details>

**Your answer:** *(write here)*

### R4. Plots that answer a question (40 min)

Make three labelled plots, with units: (a) a histogram of ozone; (b) boxplots of ozone by month; (c) ozone against temperature (°C) with points coloured by month. In two or three sentences: what does each plot show that the others do not? Is ozone's distribution symmetric?

In [ ]:
# (a) histogram of ozone: complete the labels
ggplot(aq, aes(x = Ozone)) +
  geom_histogram(bins = 20) +
  labs(title = "Daily ozone", x = "...", y = "...")

# (b) one box per month:   ggplot(aq, aes(x = factor(Month), y = Ozone)) + geom_boxplot() + labs(...)
# (c) points by month:     ggplot(aq, aes(x = TempC, y = Ozone, colour = factor(Month))) + geom_point() + labs(...)

<details><summary><b>Hint</b> (click to open)</summary>

A ggplot is built in layers joined by `+`: the data and `aes()` (which column goes on which axis, or sets the colour), then a geometry (`geom_histogram()`, `geom_boxplot()`, `geom_point()`), then `labs()` for the title and axis labels. `factor(Month)` makes ggplot treat month as five categories (one box or one colour each) instead of a number on a colour gradient; `factor(Month, labels = month.abb[5:9])` shows May–Sep. ggplot2 drops the days without ozone and warns “Removed 37 rows”: say so in your interpretation.

</details>

**Your answer:** *(write here)*

### R5. Repetition without copy-paste (20 min)

Use `sapply()` or `replicate()` to compute the median ozone of 1,000 **bootstrap resamples** (resample the observed ozone values with replacement). Report the 5 % and 95 % quantiles of these medians. What does their spread describe?

In [ ]:
oz <- aq$Ozone[!is.na(aq$Ozone)]
# meds <- replicate(1000, median(sample(oz, replace = TRUE)))

<details><summary><b>Hint</b> (click to open)</summary>

`sample(oz, replace = TRUE)` draws a resample of the same size.

</details>

**Your answer:** *(write here)*

## Part 2 — Probability

### P1. Four functions per distribution (30 min)

R names distribution functions `d` (density or probability mass), `p` (cumulative probability), `q` (quantile) and `r` (random draws). For each, write one line and say what it returns:

- (a) the density of Normal(10, 2) at 12;
- (b) P(X ≤ 12) for that normal distribution;
- (c) its 90th percentile;
- (d) 5 random draws from it;
- (e) P(Y = 3) for Binomial(10, 0.3);
- (f) P(Y ≤ 3) for that binomial;
- (g) P(N ≥ 5) for Poisson with mean 2.

In [ ]:
# dnorm(12, 10, 2); pnorm(); qnorm(); rnorm(); dbinom(); pbinom(); ppois()

<details><summary><b>Hint</b> (click to open)</summary>

P(N ≥ 5) = 1 − P(N ≤ 4). The normal in R is parameterised by its **SD**, not its variance.

</details>

**Your answer:** *(write here)*

### P2. Densities are not probabilities (20 min)

X is uniform on [0, 0.25]. (a) What is its density? (b) What are P(X = 0.1) and P(0.05 < X < 0.15)? (c) Why can a density be larger than 1 when no probability can?

In [ ]:
# dunif(0.1, 0, 0.25); punif(0.15, 0, 0.25) - punif(0.05, 0, 0.25)

**Your answer:** *(write here)*

### P3. Expectation and variance (30 min)

(a) For Y ~ Binomial(10, 0.3), compute E[Y] and Var(Y) **from the probability table** as sums, and compare with np and np(1−p). (b) For X ~ Exponential(rate 0.5), compute E[X] = ∫ x f(x) dx with `integrate()` and compare with 1/rate. (c) Check both by simulation with 100,000 draws.

In [ ]:
y <- 0:10; py <- dbinom(y, 10, 0.3)
# sum(y * py); ...
# integrate(function(x) x * dexp(x, 0.5), 0, Inf)

<details><summary><b>Hint</b> (click to open)</summary>

E[Y] = Σ y P(Y = y); Var(Y) = Σ (y − E[Y])² P(Y = y). For a continuous variable the sum becomes an integral.

</details>

**Your answer:** *(write here)*

### P4. Conditional probability and Bayes' rule (40 min)

A fictional screening test detects a condition in **95 %** of people who have it, and gives a false positive in **4 %** of people who do not. **2 %** of the tested population has the condition.

(a) Imagine 10,000 people and fill in a 2×2 table of counts: condition yes/no by test positive/negative. (b) What is P(condition | positive)? (c) The person is tested again, independently given their condition, and is again positive. Update. (d) In one sentence: why is (b) so much lower than 95 %?

In [ ]:
prior <- c(yes = 0.02, no = 0.98)
lik_pos <- c(yes = 0.95, no = 0.04)
# counts <- 10000 * ...
# posterior <- prior * lik_pos / sum(prior * lik_pos)

<details><summary><b>Hint</b> (click to open)</summary>

Of 10,000 people, 200 have the condition and 190 of them test positive; 9,800 do not and 392 of them test positive.

</details>

**Your answer:** *(write here)*

### P5. Joint, marginal and conditional (30 min)

Classify each day in `airquality` as **hot** (TempC > 25) or not, and **high ozone** (Ozone > 60 ppb) or not, using days with ozone observed. (a) Make the 2×2 table of proportions (the **joint** distribution). (b) Give the **marginal** P(high ozone) and the **conditional** P(high ozone | hot) and P(high ozone | not hot). (c) Are the two events independent? (d) Does this show that heat *causes* ozone?

In [ ]:
d <- aq[!is.na(aq$Ozone), ]
# hot <- d$TempC > 25; high <- d$Ozone > 60
# prop.table(table(hot, high))

<details><summary><b>Hint</b> (click to open)</summary>

A conditional probability is a joint probability divided by a marginal: P(high | hot) = P(high and hot) / P(hot); in R, simply `mean(high[hot])`.

</details>

**Your answer:** *(write here)*

### P6. Repeated measurements are not independent (20 min)

A person's 10-second time estimates follow Yᵢⱼ = μ + bᵢ + eᵢⱼ, where person i has a personal offset bᵢ ~ Normal(0, τ = 1) and attempt j adds noise eᵢⱼ ~ Normal(0, σ = 0.5). (a) Simulate 5,000 people with two attempts each. Estimate Var(Y) and the correlation between a person's two attempts. Compare with τ² + σ² and τ²/(τ² + σ²). (b) Why does this matter when 10 attempts per person are analysed as if they were 10 independent observations?

In [ ]:
n <- 5000; tau <- 1; sigma <- 0.5
# b <- rnorm(n, 0, tau); y1 <- 10 + b + rnorm(n, 0, sigma); y2 <- ...

**Your answer:** *(write here)*

## Part 3 — A little calculus

### C1. Normalising a density (20 min)

A density on [0, 1] is proportional to p(1 − p). (a) Find the constant c so that c·p(1 − p) integrates to 1, analytically and with `integrate()`. (b) Compare c·p(1 − p) with `dbeta(p, 2, 2)`. (c) Why is it often enough in Bayesian statistics to know a density only *up to a constant*?

In [ ]:
# integrate(function(p) p * (1 - p), 0, 1)

**Your answer:** *(write here)*

### C2. Why we work with logarithms (20 min)

(a) Simulate 1,000 values from Normal(0, 1) and compute the product of their densities with `prod(dnorm(x))`. What happens? (b) Compute the sum of log densities instead with `sum(dnorm(x, log = TRUE))`. (c) Use log rules to explain why the two contain the same information.

**Your answer:** *(write here)*

### C3. Finding a maximum (20 min)

f(p) = p⁷(1 − p)³ for p in (0, 1). (a) Find the maximising p analytically: take logs, differentiate, set to zero. (b) Check with `optimize()` and by plotting f. (c) What would change if the exponents were 70 and 30?

In [ ]:
# optimize(function(p) 7 * log(p) + 3 * log(1 - p), c(0, 1), maximum = TRUE)
# ggplot(data.frame(p = seq(0, 1, length.out = 500)), aes(x = p, y = p^7 * (1 - p)^3)) + geom_line()

**Your answer:** *(write here)*

## Part 4 — Simulation

### S1. A probability as a simulated proportion (25 min)

For X ~ Normal(10, 2), estimate P(8 < X < 12) by simulation with 100, 1,000 and 100,000 draws, each with its **Monte Carlo standard error** √(p̂(1 − p̂)/S). Compare with `pnorm`. How does the error shrink as S grows?

**Your answer:** *(write here)*

### S2. Individual values versus averages (30 min)

Draw 2,000 samples of size n = 10 and of size n = 100 from Normal(10, 2) and keep each sample mean. Compare the SD of the means with 2/√n, and plot both distributions on the same axis. Distinguish (i) the SD of individual observations, (ii) the SD of a sample mean, and (iii) the Monte Carlo error of your estimate of (ii).

In [ ]:
# m10 <- replicate(2000, mean(rnorm(10, 10, 2)))
# m100 <- ...
# Put both sets of means in one data frame, then plot them on the same axis:
# means <- tibble(n = rep(c("n = 10", "n = 100"), each = 2000), mean = c(m10, m100))
# ggplot(means, aes(x = mean)) + geom_histogram(bins = 40) + facet_wrap(~ n, ncol = 1)

**Your answer:** *(write here)*

### S3. A first likelihood (35 min)

In 10 independent trials there were **7 successes**. (a) On a grid of 1,000 values of p in (0, 1), compute the binomial likelihood `dbinom(7, 10, p)` and plot it against p. (b) Divide the likelihood by its sum over the grid, so the weights sum to 1, and compute the weighted mean of p and the probability that p > 0.5. (c) This is a Bayesian posterior under a flat prior. Which question does it answer that the maximum-likelihood estimate 0.7 alone does not?

In [ ]:
p <- seq(0.0005, 0.9995, length.out = 1000)
# lik <- dbinom(7, 10, p)
# ggplot(data.frame(p, lik), aes(x = p, y = lik)) + geom_line()

<details><summary><b>Hint</b> (click to open)</summary>

The likelihood is a function of the **parameter** p for fixed data, not a probability distribution over p — until you normalise it (with a prior).

</details>

**Your answer:** *(write here)*

## Part 5 — Your own research question

The final assignment is an exercise collection on a supplied dataset, but the aim of the course is that you can use Bayesian methods and Stan in your own research. Thinking about your own data now makes the course more useful. In 150–300 words, describe:

1. A question from your research that involves uncertainty: an estimate, a comparison, or a prediction.
2. The **outcome** you would measure, its type (count, yes/no, continuous, positive) and units.
3. The **observational unit**, and whether observations come in groups or repeated measurements (people, sites, batches, time points).
4. Something that is **already known** about the answer (a published result, an earlier analysis, expert knowledge) that could inform a prior or serve as a check.
5. One thing you are unsure how to model.

There is no right answer, and nothing is submitted. Keep your notes: they are useful when we discuss applying the methods to your own research on Day 6.

**My research question:** *(write here)*

## Part 6 — Self-check

Tick what you can do comfortably. For anything you cannot, write a short note to bring to Day 1. Nobody is expected to tick everything.

- [ ] Run a notebook from a fresh kernel with the **R (bayes)** kernel, and compile the Stan check model.
- [ ] Read, subset and summarise a data frame, handling missing values deliberately.
- [ ] Write a small function and use `sapply()` / `replicate()`.
- [ ] Make labelled plots that answer a question.
- [ ] Use the `d`/`p`/`q`/`r` functions and say what each returns.
- [ ] Explain why a density can exceed 1.
- [ ] Compute an expectation as a weighted sum or an integral.
- [ ] Apply Bayes' rule with a base rate.
- [ ] Distinguish joint, marginal and conditional probabilities.
- [ ] Explain why repeated measurements on the same unit are correlated.
- [ ] Explain why we add log densities instead of multiplying densities.
- [ ] Estimate a probability by simulation and give its Monte Carlo error.
- [ ] Explain the difference between a likelihood and a posterior.

**Bring to Day 1:** the two or three questions you most want answered.

**My questions for Day 1:** *(write here)*